In [ ]:
# =====================================================================
# CELL 1 — CONFIGURATION
# =====================================================================
import glob
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

DATA_DIR = Path(r"C:\Users\Mohimen\Desktop\lucky_case_study\data")
SPEND_FILE_PATTERN = "*marketing_spend.csv"
USERS_FILE_PATTERN = "*users.csv"
TXN_FILE_PATTERN   = "*transactions.csv"

Q3_START     = pd.Timestamp("2026-07-01")
Q3_END       = pd.Timestamp("2026-10-01")
Q3_DAYS      = (Q3_END - Q3_START).days
EXTRACT_DATE = pd.Timestamp("2026-11-05")

PAID_CHANNELS = ["Meta", "Google Ads", "TikTok"]
EXPECTED_CHANNELS = PAID_CHANNELS
EXPECTED_ROWS_PER_DAY = 3
EXPECTED_TOTAL_ROWS = Q3_DAYS * EXPECTED_ROWS_PER_DAY

DEC_BUDGET = 150_000
DEC_DAYS = 31

# Three split scenarios used throughout
CPI_OPTIMISED_SHARES = {"Meta": 0.45, "Google Ads": 0.35, "TikTok": 0.20}  # Phase 3 (CPI-only)
FINAL_SHARES         = {"Meta": 0.45, "Google Ads": 0.40, "TikTok": 0.15}  # Phase 5 (activated-based)

SATURATION_FLAT_PCT     = 5.0
SATURATION_MODERATE_PCT = 10.0

QA_NAMES_EXPLICIT = ["QA Lucky", "Test User", "test qa", "test test", "تجربه تطبيق"]

SOURCE_TO_CHANNEL = {
    "meta": "Meta", "facebook ads": "Meta", "facebook": "Meta", "fb": "Meta",
    "instagram": "Meta", "ig": "Meta",
    "google": "Google Ads", "google ads": "Google Ads", "adwords": "Google Ads", "gads": "Google Ads",
    "tiktok": "TikTok", "tt": "TikTok",
    "referral": "Referral", "ref": "Referral", "invite": "Referral",
    "organic": "Organic", "(direct)": "Direct",
}


pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.2f}".format)
print("✓ Config loaded")

✓ Config loaded


In [9]:
# =====================================================================
# PHASE 1 — CPI from marketing_spend (platform's reported view)
# =====================================================================
spend = pd.read_csv(glob.glob(str(DATA_DIR / SPEND_FILE_PATTERN))[0], parse_dates=["date"])

# Validation
assert len(spend) == EXPECTED_TOTAL_ROWS, f"Row count {len(spend)}"
assert spend.duplicated(["date", "channel"]).sum() == 0, "Dup (date, channel)"
assert (spend.groupby("date").size() == EXPECTED_ROWS_PER_DAY).all(), "Day row count"
assert set(spend.channel) == set(EXPECTED_CHANNELS), "Unexpected channels"

q3_spend = spend.query("date >= @Q3_START and date < @Q3_END").copy()

cpi = (q3_spend.groupby("channel", as_index=False)
       .agg(spend_egp=("spend_egp", "sum"),
            reported_installs=("platform_reported_installs", "sum"))
       .assign(cpi=lambda d: (d.spend_egp / d.reported_installs).round(2)))

total_spend_q3    = q3_spend.spend_egp.sum()
total_installs_q3 = q3_spend.platform_reported_installs.sum()

print(f"Total Q3 spend    : {total_spend_q3:,.2f} EGP")
print(f"Total Q3 installs : {total_installs_q3:,}")
print(f"Blended CPI       : {total_spend_q3/total_installs_q3:.2f} EGP\n")
print("── CPI per channel ──")
display(cpi.set_index("channel"))

CPI_PER_CHANNEL = cpi.set_index("channel")["cpi"].to_dict()

Total Q3 spend    : 356,538.71 EGP
Total Q3 installs : 6,343
Blended CPI       : 56.21 EGP

── CPI per channel ──


,spend_egp,reported_installs,cpi
channel,,,
Google Ads,"168,517.02",1803,93.46
Meta,"145,369.33",3355,43.33
TikTok,"42,652.36",1185,35.99


In [10]:
# =====================================================================
# PHASE 2 — Cost per signup (reconciled to internal users table)
# =====================================================================
users = pd.read_csv(glob.glob(str(DATA_DIR / USERS_FILE_PATTERN))[0],
                    parse_dates=["signup_ts", "kyc_verified_ts"])

# QA removal (29 users)
qa_mask = users.full_name.str.strip().isin(QA_NAMES_EXPLICIT)
qa_removed = int(qa_mask.sum())
users = users[~qa_mask].reset_index(drop=True)

# Channel bucketing
def bucket(src):
    s = str(src).strip().lower()
    if s in ("", "nan", "none"):
        return "Unattributed"
    return SOURCE_TO_CHANNEL.get(s, "Unknown")
users["channel"] = users.utm_source.map(bucket)
assert (users.channel == "Unknown").sum() == 0, "Unmapped source"

users_q3 = users.query("signup_ts >= @Q3_START and signup_ts < @Q3_END")

paid_signups = (users_q3[users_q3.channel.isin(PAID_CHANNELS)]
                .groupby("channel").size()
                .reindex(PAID_CHANNELS).fillna(0).astype(int))
n_unattr = int((users_q3.channel == "Unattributed").sum())

spend_by_channel = cpi.set_index("channel").loc[PAID_CHANNELS, "spend_egp"]

cpsu = pd.DataFrame({
    "spend_egp": spend_by_channel,
    "paid_signups": paid_signups,
    "cost_per_signup": (spend_by_channel / paid_signups).round(2),
})

print(f"QA users removed   : {qa_removed}")
print(f"Users after removal: {len(users):,}")
print(f"Unattributed       : {n_unattr:,}\n")
print("── Cost per signup ──")
display(cpsu)

COST_PER_SIGNUP = cpsu["cost_per_signup"].to_dict()

QA users removed   : 29
Users after removal: 6,201
Unattributed       : 765

── Cost per signup ──


,spend_egp,paid_signups,cost_per_signup
channel,,,
Meta,"145,369.33",2100,69.22
Google Ads,"168,517.02",1430,117.84
TikTok,"42,652.36",650,65.62


In [11]:
# =====================================================================
# PHASE 3 — Saturation quartiles + CPI-only split (45/35/20)
# =====================================================================
q3_daily = q3_spend.copy()
q3_daily["cpi_daily"] = q3_daily.spend_egp / q3_daily.platform_reported_installs

def classify(delta_pct):
    if delta_pct < SATURATION_FLAT_PCT:
        return "Not saturated — scales well"
    if delta_pct < SATURATION_MODERATE_PCT:
        return "Partially saturated — scales ok"
    return "Saturated — will not scale"

sat_rows = []
for ch in PAID_CHANNELS:
    sub = q3_daily[q3_daily.channel == ch].copy()
    sub["spend_q"] = pd.qcut(sub.spend_egp, 4,
                             labels=["Lowest-spend 25%", "Low-mid", "High-mid", "Highest-spend 25%"])
    g = sub.groupby("spend_q", observed=True).agg(
        median_cpi=("cpi_daily", "median"),
    )
    low, high = g.iloc[0], g.iloc[-1]
    delta_pct = 100 * (high.median_cpi - low.median_cpi) / low.median_cpi
    sat_rows.append({
        "channel": ch,
        "low_spend_cpi": round(low.median_cpi, 2),
        "high_spend_cpi": round(high.median_cpi, 2),
        "cpi_change_pct": round(delta_pct, 1),
        "bucket": classify(delta_pct),
    })

saturation = pd.DataFrame(sat_rows).set_index("channel")

q3_shares = (q3_daily.groupby("channel").spend_egp.sum() / q3_daily.spend_egp.sum()).to_dict()

cpi_split = pd.DataFrame({
    "q3_share_pct":      {ch: round(100 * q3_shares[ch], 1) for ch in PAID_CHANNELS},
    "cpi_split_pct":     {ch: round(100 * CPI_OPTIMISED_SHARES[ch], 1) for ch in PAID_CHANNELS},
    "cpi_split_egp":     {ch: round(DEC_BUDGET * CPI_OPTIMISED_SHARES[ch], 0) for ch in PAID_CHANNELS},
})

print("── Saturation (quartile analysis) ──")
display(saturation)
print("\n── CPI-only recommendation (45/35/20) ──")
display(cpi_split)

── Saturation (quartile analysis) ──


,low_spend_cpi,high_spend_cpi,cpi_change_pct,bucket
channel,,,,
Meta,42.85,43.63,1.80,Not saturated — scales well
Google Ads,84.11,98.57,17.20,Saturated — will not scale
TikTok,34.93,37.06,6.10,Partially saturated — scales ok



── CPI-only recommendation (45/35/20) ──


,q3_share_pct,cpi_split_pct,cpi_split_egp
Meta,40.80,45.00,"67,500.00"
Google Ads,47.30,35.00,"52,500.00"
TikTok,12.00,20.00,"30,000.00"


In [12]:
# =====================================================================
# PHASE 4 — Cost per activated user (README definition)
# Activated = KYC verified + ≥1 successful deposit ≤7d of signup
# =====================================================================
txn = pd.read_csv(glob.glob(str(DATA_DIR / TXN_FILE_PATTERN))[0], parse_dates=["txn_ts"])

# DQ: dedupe txn_id, count orphans
n_dup_txn    = int(txn.txn_id.duplicated().sum())
txn_dedup    = txn.drop_duplicates("txn_id", keep="first")
n_orphan_txn = int((~txn_dedup.user_id.isin(users.user_id)).sum())

# Activated users
txn_clean = (txn_dedup
             .query("status == 'success' and txn_type == 'deposit'")
             .merge(users[["user_id", "signup_ts", "kyc_verified_ts", "channel"]],
                    on="user_id", how="inner"))
txn_clean["days_since"] = (txn_clean.txn_ts - txn_clean.signup_ts).dt.total_seconds() / 86400

activated_users = (txn_clean
                   .query("days_since >= 0 and days_since <= 7")
                   .query("kyc_verified_ts.notna()")
                   .drop_duplicates("user_id"))

paid_activated = (activated_users[activated_users.channel.isin(PAID_CHANNELS)]
                  .groupby("channel").size()
                  .reindex(PAID_CHANNELS).fillna(0).astype(int))

cpa = pd.DataFrame({
    "spend_egp":        spend_by_channel,
    "signups":          paid_signups,
    "activated_users":  paid_activated,
    "activation_rate_pct": (100 * paid_activated / paid_signups).round(1),
    "cost_per_activated": (spend_by_channel / paid_activated).round(2),
})

print(f"── Transaction DQ ──")
print(f"Duplicate txn_id rows : {n_dup_txn:,}")
print(f"Orphan txn rows       : {n_orphan_txn:,}")
print(f"Activated users (KYC + deposit ≤7d): {len(activated_users):,}\n")
print("── Cost per activated user ──")
display(cpa)

COST_PER_ACTIVATED = cpa["cost_per_activated"].to_dict()

── Transaction DQ ──
Duplicate txn_id rows : 577
Orphan txn rows       : 672
Activated users (KYC + deposit ≤7d): 2,362

── Cost per activated user ──


,spend_egp,signups,activated_users,activation_rate_pct,cost_per_activated
channel,,,,,
Meta,"145,369.33",2100,522,24.90,278.49
Google Ads,"168,517.02",1430,730,51.00,230.85
TikTok,"42,652.36",650,132,20.30,323.12


In [15]:
# =====================================================================
# PHASE 5 — Final recommendation (45/40/15) + master comparison
# =====================================================================
scenarios = {
    "1. Q3 actual (356,539 EGP)":         (q3_shares,           total_spend_q3),
    "2. 150K @ Q3 shares (do-nothing)":   (q3_shares,           DEC_BUDGET),
    "3. 150K @ 45/35/20 (CPI-based)":     (CPI_OPTIMISED_SHARES, DEC_BUDGET),
    "4. 150K @ 45/40/15 (FINAL)":         (FINAL_SHARES,        DEC_BUDGET),
}

# ── Table A — Scenario summary ───────────────────────────────────────
summary_rows = []
for name, (sh, budget) in scenarios.items():
    inst = sum(budget * sh[ch] / CPI_PER_CHANNEL[ch]     for ch in PAID_CHANNELS)
    sign = sum(budget * sh[ch] / COST_PER_SIGNUP[ch]     for ch in PAID_CHANNELS)
    act  = sum(budget * sh[ch] / COST_PER_ACTIVATED[ch]  for ch in PAID_CHANNELS)
    summary_rows.append({
        "scenario":           name,
        "budget_egp":         int(round(budget)),
        "meta_pct":           round(100 * sh["Meta"],       1),
        "google_pct":         round(100 * sh["Google Ads"], 1),
        "tiktok_pct":         round(100 * sh["TikTok"],     1),
        "installs":           int(round(inst)),
        "signups":            int(round(sign)),
        "activated":          int(round(act)),
    })

scenario_summary = pd.DataFrame(summary_rows)

# ── Table B — Per-channel detail ─────────────────────────────────────
detail_rows = []
for name, (sh, budget) in scenarios.items():
    for ch in PAID_CHANNELS:
        egp = budget * sh[ch]
        detail_rows.append({
            "scenario":          name,
            "channel":           ch,
            "spend_egp":         int(round(egp)),
            "share_pct":         round(100 * sh[ch], 1),
            "cpi":               CPI_PER_CHANNEL[ch],
            "cost_per_signup":   COST_PER_SIGNUP[ch],
            "cost_per_activated": COST_PER_ACTIVATED[ch],
            "installs":          int(round(egp / CPI_PER_CHANNEL[ch])),
            "signups":           int(round(egp / COST_PER_SIGNUP[ch])),
            "activated":         int(round(egp / COST_PER_ACTIVATED[ch])),
        })

channel_detail = pd.DataFrame(detail_rows)

print("═══ TABLE A — Scenario summary ═══")
display(scenario_summary.set_index("scenario"))
print("\n═══ TABLE B — Per-channel detail ═══")
display(channel_detail.set_index(["scenario", "channel"]))

═══ TABLE A — Scenario summary ═══


,budget_egp,meta_pct,google_pct,tiktok_pct,installs,signups,activated
scenario,,,,,,,
"1. Q3 actual (356,539 EGP)",356539,40.80,47.30,12.00,6343,4180,1384
2. 150K @ Q3 shares (do-nothing),150000,40.80,47.30,12.00,2669,1759,582
3. 150K @ 45/35/20 (CPI-based),150000,45.00,35.00,20.00,2953,1878,563
4. 150K @ 45/40/15 (FINAL),150000,45.00,40.00,15.00,2825,1827,572



═══ TABLE B — Per-channel detail ═══


spend_egp  share_pct   cpi  cost_per_signup  cost_per_activated  installs  signups  activated
scenario                         channel                                                                                                  
1. Q3 actual (356,539 EGP)       Meta           145369      40.80 43.33            69.22              278.49      3355     2100        522
                                 Google Ads     168517      47.30 93.46           117.84              230.85      1803     1430        730
                                 TikTok          42652      12.00 35.99            65.62              323.12      1185      650        132
2. 150K @ Q3 shares (do-nothing) Meta            61159      40.80 43.33            69.22              278.49      1411      884        220
                                 Google Ads      70897      47.30 93.46           117.84              230.85       759      602        307
                                 TikTok          17944      12.00 35.99            65.62              323.12       499      273         56
3. 150K @ 45/35/20 (CPI-based)   Meta            67500      45.00 43.33            69.22              278.49      1558      975        242
                                 Google Ads      52500      35.00 93.46           117.84              230.85       562      446        227
                                 TikTok          30000      20.00 35.99            65.62              323.12       834      457         93
4. 150K @ 45/40/15 (FINAL)       Meta            67500      45.00 43.33            69.22              278.49      1558      975        242
                                 Google Ads      60000      40.00 93.46           117.84              230.85       642      509        260
                                 TikTok          22500      15.00 35.99            65.62              323.12       625      343         70

# Marketing Analysis — Visual Summary

## What we did (5 phases)

| Phase | Question | Input | Output |
|---|---|---|---|
| **1** | What does the platform say CPI is? | `marketing_spend` | CPI per channel |
| **2** | Does that hold against real signups? | `users.signup_ts` | Cost per signup |
| **3** | Do channels scale? What split if CPI-only? | Phase 1 + 2 | Saturation buckets + 45/35/20 |
| **4** | What's the metric that drives revenue? | `transactions` + README activated def | Cost per activated user |
| **5** | What's the final split? | Phases 1–4 | **45/40/15** + master comparison |

---

## Headline numbers (Q3, 356,539 EGP)

| Metric | Meta | Google Ads | TikTok |
|---|---:|---:|---:|
| **CPI** | 43.33 | 93.46 | 35.99 |
| **Cost per signup** | 69.22 | 117.84 | 65.62 |
| **Cost per activated** | 278.49 | **230.85** | 323.12 |
| **Activation rate** | 24.9% | **51.0%** | 20.3% |
| **Saturation (CPI Δ)** | +1.8% | +17.2% | +6.1% |
| **Saturation (activated Δ)** | +10.1% | +7.6% | unstable |

---

## The story arc

**Phase 1–2 (CPI & signups):** Meta wins.
- Meta 43 CPI / 69 per signup vs Google 93 CPI / 118 per signup
- Direction: shift toward Meta

**Phase 4 (activated):** **Ranking REVERSES.**
- Google: 230.85 per activated user
- Meta: 278.49 per activated user
- **Why:** Google signups deposit at 51% vs Meta's 25% — 2× the conversion

**Phase 5 (final split):** 45/40/15.
- Google restored from 35% to 40% — because it produces real depositing customers
- TikTok trimmed to 15% — test size, not commitment
- Meta holds at 45% — flat CPI, diversification hedge

---

## The reversal in one sentence

> *"On CPI and cost per signup, Meta is cheapest — but on cost per activated user (KYC + deposit ≤7d, per README), Google is cheapest. Google's users actually deposit; Meta's installs are cheap and empty."*

---

## Final split: 45 / 40 / 15

| Channel | Q3 share | Dec share | Dec EGP | Why |
|---|---:|---:|---:|---|
| Meta | 40.8% | **45%** | 67,500 | Volume + flat CPI (diversification) |
| Google | 47.3% | **40%** | 60,000 | **Cheapest per activated** — keep high |
| TikTok | 12.0% | **15%** | 22,500 | Test-sized, not committed |

---

## Impact comparison (same 150K budget)

| Scenario | Installs | Signups | **Activated** |
|---|---:|---:|---:|
| Q3 shares @ 150K (do-nothing) | 2,672 | 1,758 | **582** |
| 45/35/20 (CPI-optimised) | 2,953 | 1,883 | **563** ← best on vanity, worst on real |
| **45/40/15 (FINAL)** | **2,825** | **1,827** | **573** |

**The lesson:** CPI-optimised 45/35/20 gains ~290 more installs but **loses 19 activated users** vs the final split. **Vanity metric up, real metric down — the classic trap.**

---

## Key decisions to defend in the debrief

1. **QA exclusion** (29 users from `full_name` list) — 0 impact on paid channels, pure hygiene
2. **Attribution handling** — 765 unattributed (12.3%) excluded from strict, sensitivity tested in 4 scenarios
3. **Saturation** — quartile-based, no regression needed
4. **Activation** — README definition enforced (`KYC AND deposit ≤7d`)
5. **Why 45/40/15 not 45/35/20** — activated metric reverses the CPI ranking
6. **Why not aggressive Google (50%+)** — saturation risk + concentration risk + Search-brand ceiling

---

## What we know vs what we don't

**Defensible:**
- Google is cheapest per activated user across attribution scenarios
- Meta is cheapest on CPI and signup (the intermediary metrics)
- All three channels are within Q3's observed spend envelope at 150K

**Not defensible (needs experiment):**
- Exact optimal split (no way to know without measuring marginal cost at 2× spend)
- Whether Google's activation advantage holds at 50%+ share
- Whether TikTok scales (weak/unstable signal)

**Next step:** December experiment with 45/40/15 as baseline + controlled spend lifts on Google and TikTok to measure elasticity directly.